In [1]:
####################################
#ENVIRONMENT SETUP

In [2]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle
import glob

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [3]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [4]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "ERA5_Data")
dataType = "ERA5Comparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Data Directory:           /glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/ERA5_Data



In [5]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [10]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

spinup_hours = "24"
# spinup_hours = "12"

# RunType = ("TRACER","MOIST","NSSL",spinup_hours)
RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

# RunType = ("TRACER","MOIST","TEMPO",spinup_hours)
RunType = ("TRACER","DRY","TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

Found 289/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/DRY/MPAS-Model_NSSL/model_run_spinup24hrs/history_cartesian/history.2022-06-08_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/DRY/MPAS-Model_NSSL/model_run_spinup24hrs/diag_cartesian/diag.2022-06-08_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           DRY
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-08 to 2022-06-11
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:289
 # Diag Files:   289
 # Time Steps:   289
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/DRY/MPAS-Model_NSSL/model_run_spinup24hrs
 Static File:    TRACER_regional5250_scale

In [11]:
#Importing Plotting Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_Plotting import ContourPlotting_Class

In [12]:
#Importing DataSaving Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [13]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class

In [14]:
###############
#JOB ARRAY SETUP

In [15]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [16]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetNumElements():
    num_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return num_elements
loop_elements = GetNumElements()

Running timesteps from 0:14 



In [17]:
########################
#DATA INFORMATION

In [18]:
# ERA5 atmospheric surface analysis [netCDF4]
# https://gdex.ucar.edu/datasets/d633000/
# https://gdex.ucar.edu/datasets/d633000/filelist/29/?fl=glade

In [19]:
##########################
#PLOTTING FUNCTIONS

In [20]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

def ConvertDatetime64ToTimeTitle(dt64):
    """
    Converts numpy.datetime64 to 'YYYY-MM-DD HH:MM:SS' format.
    """
    return pd.to_datetime(dt64).strftime('%Y-%m-%d %H:%M:%S')

In [21]:
#Getting TimeData
def GetData(t):
    timeString = ModelData_NSSL.timeStrings[t]
    timeString_datetime = ConvertTimeStringtoDateTime(timeString)
    
    #Loading Model Radar
    modelData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)["mslp"]
    modelData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)["mslp"]
    modelTimeTitle = ConvertTimeStringtoTimeTitle(timeString)
    
    #Loading Observational Radar
    ERA5Data_alltimes = ERA5DataLoading_Class.LoadERA5Data(timeString, ModelData_NSSL, DirectoryManager)
    ERA5Data = ERA5DataLoading_Class.SelectNearestERA5Time(ERA5Data_alltimes, timeString)
    ERA5TimeTitle = ConvertDatetime64ToTimeTitle(ERA5Data.time.data)

    return (modelData_NSSL,modelData_TEMPO,modelTimeTitle,
            ERA5Data, ERA5TimeTitle,
            timeString)

In [29]:
ERA5DataLoading_Class.GetERA5FilePath(DirectoryManager,ModelData_NSSL.timeStrings[100])

'/glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/globus_download/d633000/e5.oper.an.sfc/202206/e5.oper.an.sfc.128_151_msl.ll025sc.2022060100_2022063023.nc'

In [22]:
##########################
#PLOTTING FUNCTIONS

In [23]:
def ComputeOrLoadClims(ModelData_NSSL, ModelData_TEMPO, GetData, outFilePath="clim_dict.pkl"):
    """
    Computes min/max for 3 datasets across all time steps and saves/loads them to/from a .pkl file.
    """

    # If file exists, load and return it
    if os.path.exists(outFilePath):
        with open(outFilePath, "rb") as f:
            clim_dict = pickle.load(f)
        print(f"Loaded cached clim dictionary from: {outFilePath}")
        return clim_dict

    # Otherwise compute it
    mins_nssl, maxs_nssl = [], []
    mins_tempo, maxs_tempo = [], []
    mins_era5, maxs_era5 = [], []

    for t in tqdm(range(ModelData_NSSL.Ntime), desc="Computing clim bounds"):
        modelData_NSSL, modelData_TEMPO, _, ERA5Data, _, _ = GetData(t)

        # Convert to numpy arrays and flatten
        mins_nssl.append(float(modelData_NSSL.min()))
        maxs_nssl.append(float(modelData_NSSL.max()))

        mins_tempo.append(float(modelData_TEMPO.min()))
        maxs_tempo.append(float(modelData_TEMPO.max()))

        mins_era5.append(float(ERA5Data.min()))
        maxs_era5.append(float(ERA5Data.max()))

    # Create dictionary and save
    clim_dict = {
        'nssl': {'min': mins_nssl, 'max': maxs_nssl},
        'tempo': {'min': mins_tempo, 'max': maxs_tempo},
        'era5': {'min': mins_era5, 'max': maxs_era5},
    }

    with open(outFilePath, "wb") as f:
        pickle.dump(clim_dict, f)
        print(f"Saved clim dictionary to: {outFilePath}")

    return clim_dict

def GetClim(clim_dict):
    # To get global colorbar limits:
    vmin = min(min(clim_dict['nssl']['min']),
               min(clim_dict['tempo']['min']),
               min(clim_dict['era5']['min']))
    
    vmax = max(max(clim_dict['nssl']['max']),
               max(clim_dict['tempo']['max']),
               max(clim_dict['era5']['max']))
    clim = (vmin,vmax)
    return clim

In [24]:
def MakePlot(modelData_NSSL,modelData_TEMPO,modelTimeTitle,
             ERA5Data, ERA5TimeTitle,
             timeString,
             clim=(None,None)):
    multiplier = 1/1e2
    
    fig, axes = ContourPlotting_Class.CreateMapAxes(nrows=1,ncols=3,
                                                  figsize=(16,8))
    
    #Plotting ModelRadar
    #nssl
    axis = axes[0,0]
    lat = modelData_NSSL['latitude']
    lon = modelData_NSSL['longitude']
    contourPlot = ContourPlotting_Class.PlotContourPlot(axis,lat,lon,modelData_NSSL,
                                                        dataName="NSSL", timeTitle = modelTimeTitle,
                                                        multiplier = multiplier,
                                                        clim=clim)

    #tempo
    axis = axes[0,2]
    lat = modelData_TEMPO['latitude']
    lon = modelData_TEMPO['longitude']
    contourPlot = ContourPlotting_Class.PlotContourPlot(axis,lat,lon,modelData_TEMPO,
                                                        dataName="TEMPO", timeTitle = modelTimeTitle,
                                                        multiplier = multiplier,
                                                        clim=clim)

    
    #Plotting Observational Radar
    #mrms data
    axis = axes[0,1]
    lat = ERA5Data['latitude'].data
    lon = ERA5Data['longitude'].data    
    contourPlot = ContourPlotting_Class.PlotContourPlot(axis,lat,lon,ERA5Data,
                                                        dataName="ERA5", timeTitle = modelTimeTitle,
                                                        multiplier = multiplier,
                                                        clim=clim)


    #Shared Colorbar
    colorbarTitle = f"{modelData_TEMPO.name.upper()} "+fr"($hPa$)"
    colorBar = ContourPlotting_Class.AddSharedColorbar(fig, contourPlot, colorbarTitle = colorbarTitle)

    return fig

In [25]:
def GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData_1.region}_{ModelData_1.case}_{ModelData_1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    
    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory)
    return outputFilePath

In [26]:
def SaveFigure(fig, ModelData_1,ModelData_2, timeString):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"ERA5Comparison_{ModelData_1.mpType}vsERA5vs{ModelData_2.mpType}_{ModelData_1.spinup_hours}_{timeString}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
##########################
#PLOTTING

In [ ]:
useClim = True
if useClim == True:
    clim_dict = ComputeOrLoadClims(ModelData_NSSL, ModelData_TEMPO, GetData,
                               outFilePath="ERA5Comparison_Surface_clims.pkl")
    clim = GetClim(clim_dict)
    climString = ""
else:
    clim = (None,None)
    climString = "_noclim"

In [ ]:
loop_elements = range(ModelData_NSSL.Ntime)
for t in tqdm(loop_elements, desc="Processing"):
    [modelData_NSSL,modelData_TEMPO,modelTimeTitle,
     ERA5Data, ERA5TimeTitle,
     timeString]=GetData(t)
    fig = MakePlot(modelData_NSSL,modelData_TEMPO,modelTimeTitle,
                   ERA5Data, ERA5TimeTitle,
                   timeString,
                   clim=clim)
    SaveFigure(fig, ModelData_NSSL,ModelData_TEMPO, timeString)

In [ ]:
#################
#MAKING ANIMATION
ANIMATE=False #keep false when running with bash code
# ANIMATE=True

In [ ]:
if ANIMATE==True:    
    #Importing AnimationPlotting_Class
    sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
    from CLASSES_PlottingModelData import AnimationPlotting_Class

In [ ]:
def GetVariableInputFiles(ModelData_1,ModelData_2, outputPlottingDirectory):
    filePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    fileName = f"*.png"
    filePattern = DirectoryManager.GetOutputFile(outputPlottingDirectory, filePath, fileName)
    print(filePattern)
    filePaths = DirectoryManager.GetSortedFileListByTimestamp(filePattern)
    return filePaths

def GetPlottingFileName(ModelData_1,ModelData_2, outputPlottingDirectory, filePaths, extension="mp4"):
    filePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)

    splits = os.path.basename(filePaths[0]).split('_')
    plottingFileName = f"{splits[0]}_{splits[1]}{climString}.{extension}"
    
    plottingFilePath = DirectoryManager.GetOutputFile(outputPlottingDirectory, filePath, plottingFileName)
    return plottingFilePath

In [ ]:
# PNGtoMP4 VERSION
if ANIMATE==True:    
    # Setting up output file
    print("getting file information")
    imageFiles = GetVariableInputFiles(ModelData_NSSL,ModelData_TEMPO, outputPlottingDirectory)
    plottingFilePath = GetPlottingFileName(ModelData_NSSL,ModelData_TEMPO, outputPlottingDirectory, imageFiles, extension="mp4")

     # running animation
    fps = AnimationPlotting_Class.CalculateFPS(num_frames=ModelData_NSSL.Ntime, time_interval_minutes=15, desired_duration_min=1)
    AnimationPlotting_Class.PNGsToMP4(imageFiles, plottingFilePath, fps=fps)